# DataCrunch #2 — expériences (recherche)

## Ce que j'ai trouvé en analysant la cible

```
exactement 0  : 87.18 %   ← la majorité des lignes n'a AUCUNE information de classement
exactement +1 :  4.30 %
exactement -1 :  4.16 %
continu dans [-1, 1] ailleurs
non nul       : 12.82 %
```

**Conséquence :** un modèle entraîné naïvement sur `target` passe 87 % de sa capacité
à apprendre à prédire zéro. C'est du gaspillage.

Et : `id` est **unique par moon** (250 999 ids pour 250 999 lignes) — ce n'est pas un
actif suivi, donc aucune feature « historique de l'actif » n'est possible.

## Les idées testées ici

| # | Idée | Pourquoi |
|---|---|---|
| A | LightGBM brut (référence) | ce que tout le monde fait |
| B | **Modèle à deux étages** : P(cible≠0) × E[cible \| ≠0] | exploite la structure 87/13 |
| C | Pondération : les lignes non nulles comptent plus | variante simple de B |
| D | Moins de features (top 60) | quand le signal est faible, moins c'est mieux |
| E | Transformation normale des features (uniformes → gaussiennes) | les features sont des rangs uniformes |
| F | Ensemble B + D | combiner les deux leviers |
| G | Classification binaire ±1 uniquement | capter le signal « gros mouvement » |
| — | **hasard** | témoin obligatoire |

Chaque expérience est jugée en walk-forward sur les 20 derniers moons, métrique =
corrélation de Pearson cross-sectionnelle (le score exact du concours).


In [ ]:
!pip install -q lightgbm 2>&1 | tail -1
import numpy as np, pandas as pd, pyarrow.parquet as pq, json, os, gc, time, glob, warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb
from scipy.stats import norm
T0 = time.time()
def el(): return f"{time.time()-T0:.0f}s"
BASE = os.path.dirname(glob.glob("/kaggle/input/**/X.reduced.parquet", recursive=True)[0])
y = pd.read_parquet(f"{BASE}/y.reduced.parquet").sort_values(["moon","id"]).reset_index(drop=True)
moon = y["moon"].to_numpy(np.int32); tgt = y["target"].to_numpy(np.float32)
uniq, starts, counts = np.unique(moon, return_index=True, return_counts=True)
gm = np.repeat(np.arange(len(uniq)), counts)
pf = pq.ParquetFile(f"{BASE}/X.reduced.parquet")
feats = [c for c in [pf.metadata.schema.column(i).name for i in range(pf.metadata.num_columns)] if c.startswith("Feature_")]
print(f"{len(y)} lignes · {len(uniq)} moons · {len(feats)} features · {el()}")
print(f"cible non nulle : {(tgt != 0).mean()*100:.2f} %")


In [ ]:
# criblage (nécessaire pour les variantes top-K)
mt = np.add.reduceat(tgt, starts)/counts; tc = tgt - mt[gm]
tc /= np.sqrt(np.add.reduceat(tc**2, starts))[gm]
def xc_(x): return x - (np.add.reduceat(x, starts, axis=0)/counts[:,None])[gm]

abss, signs = [], []
B = 100
for i in range(0, len(feats), B):
    blk = feats[i:i+B]
    x = pf.read(columns=blk).to_pandas()[blk].to_numpy(np.float32)
    xc = xc_(x); del x; gc.collect()
    nx = np.sqrt(np.add.reduceat(xc**2, starts, axis=0)); nx[nx==0]=np.nan
    c = np.add.reduceat(xc*tc[:,None], starts, axis=0)/nx
    abss.append(np.nanmean(np.abs(c), axis=0)); signs.append(np.nanmean(np.sign(c), axis=0))
    del xc, nx, c; gc.collect()
rank = pd.DataFrame({"feature":feats,"abs_corr":np.concatenate(abss),
                     "signe":np.concatenate(signs)}).sort_values("abs_corr", ascending=False)
print("criblage fait ·", el(), "| top :", rank.iloc[0]['feature'], round(rank.iloc[0]['abs_corr'],4))


In [ ]:
# matrice complète (les 1150 features : on en a besoin pour tester top-K)
X = np.empty((len(moon), len(feats)), dtype=np.float32)
for i in range(0, len(feats), 100):
    sub = feats[i:i+100]
    b = pf.read(columns=sub).to_pandas()[sub].to_numpy(np.float32)
    X[:, i:i+len(sub)] = b; del b; gc.collect()
for j in range(0, len(feats), 100):
    X[:, j:j+100] = xc_(X[:, j:j+100]); gc.collect()
np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
col_of = {f:i for i,f in enumerate(feats)}
print("matrice", X.shape, f"{X.nbytes/1e9:.2f} Go ·", el())


In [ ]:
EMBARGO, N_TEST = 4, 20
test_moons = list(uniq[-N_TEST:])
TOPK300 = [col_of[f] for f in rank.head(300)["feature"]]
TOPK60  = [col_of[f] for f in rank.head(60)["feature"]]

def pear(a, b):
    a = a-a.mean(); b = b-b.mean()
    d = np.sqrt((a*a).sum()*(b*b).sum())
    return float((a*b).sum()/d) if d > 0 else 0.0

def split(m):
    te = np.flatnonzero(moon == m); tr = np.flatnonzero(moon <= m - EMBARGO)
    return tr, te

res = {}
def add(name, m, p, yte):
    res.setdefault(name, []).append(pear(p, yte))

print("walk-forward sur", len(test_moons), "moons\n")
for m in test_moons:
    tr, te = split(m)
    if len(tr) < 5000 or len(te) < 30: continue
    yte = tgt[te]; nz = tgt[tr] != 0

    add("hasard", m, np.random.default_rng(int(m)).normal(size=len(te)), yte)

    # A — référence : LightGBM brut, 300 features
    g = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.04, num_leaves=15,
        min_child_samples=300, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
        reg_lambda=5.0, n_jobs=-1, verbose=-1)
    Xa = X[:, TOPK300]
    g.fit(Xa[tr], tgt[tr]); add("A lgbm brut", m, g.predict(Xa[te]), yte)

    # B — deux étages : P(non nul) × E[cible | non nul]
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.04, num_leaves=15,
        min_child_samples=300, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
        reg_lambda=5.0, n_jobs=-1, verbose=-1)
    clf.fit(Xa[tr], nz.astype(int))
    p_nz = clf.predict_proba(Xa[te])[:, 1]
    reg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.04, num_leaves=15,
        min_child_samples=100, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
        reg_lambda=5.0, n_jobs=-1, verbose=-1)
    reg.fit(Xa[tr][nz], tgt[tr][nz])
    add("B deux etages", m, p_nz * reg.predict(Xa[te]), yte)

    # C — pondération des lignes non nulles
    gw = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.04, num_leaves=15,
        min_child_samples=300, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
        reg_lambda=5.0, n_jobs=-1, verbose=-1)
    gw.fit(Xa[tr], tgt[tr], sample_weight=np.where(nz, 8.0, 1.0))
    add("C pondere x8", m, gw.predict(Xa[te]), yte)

    # D — top 60 features
    Xd = X[:, TOPK60]
    gd = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.04, num_leaves=15,
        min_child_samples=300, colsample_bytree=0.6, subsample=0.8, subsample_freq=1,
        reg_lambda=5.0, n_jobs=-1, verbose=-1)
    gd.fit(Xd[tr], tgt[tr]); add("D top60", m, gd.predict(Xd[te]), yte)

    # E — transformation normale des rangs uniformes
    Xe_tr = Xa[tr].copy(); Xe_te = Xa[te].copy()
    mu = Xe_tr.mean(0); sd = Xe_tr.std(0); sd[sd==0] = 1
    Ztr = (Xe_tr - mu)/sd; Zte = (Xe_te - mu)/sd
    ge = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.04, num_leaves=15,
        min_child_samples=300, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
        reg_lambda=5.0, n_jobs=-1, verbose=-1)
    ge.fit(Ztr, tgt[tr]); add("E gaussien", m, ge.predict(Zte), yte)

    # G — direction ±1 seulement
    big = (np.abs(tgt[tr]) > 0.5).astype(int)
    if big.sum() > 500:
        gc_ = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.04, num_leaves=15,
            min_child_samples=200, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
            reg_lambda=5.0, n_jobs=-1, verbose=-1)
        gc_.fit(Xa[tr], big)
        add("G signe ±1", m, gc_.predict_proba(Xa[te])[:, 1], yte)

    print(f"  moon {m} · {el()}", flush=True)
    del g, clf, reg, gw, gd, ge; gc.collect()


In [ ]:
print("="*66); print(" RÉSULTATS — hors échantillon, 20 moons"); print("="*66)
print(f"{'expérience':<20}{'moyenne':>10}{'médiane':>10}{'%>0':>7}{'t-stat':>8}")
print("-"*66)
rows = []
for k, v in sorted(res.items(), key=lambda kv: -np.mean(kv[1])):
    v = np.array(v); t = v.mean()/(v.std(ddof=1)/np.sqrt(len(v))) if len(v)>1 else 0
    rows.append((k, v.mean(), np.median(v), (v>0).mean()*100, t))
    print(f"{k:<20}{v.mean():>+10.4f}{np.median(v):>+10.4f}{(v>0).mean()*100:>6.0f}%{t:>8.2f}")
base = np.mean(res["A lgbm brut"]); rnd = np.mean(res["hasard"])
print("-"*66)
best = max((r for r in rows if r[0] != "hasard"), key=lambda r: r[1])
print(f"\nmeilleure : {best[0]}  {best[1]:+.4f}  (t={best[4]:.2f})")
print(f"référence A : {base:+.4f} | hasard : {rnd:+.4f}")
print(f"gain vs référence A : {best[1]-base:+.4f}")
json.dump({k:[float(x) for x in v] for k,v in res.items()}, open("exp_results.json","w"), indent=2)
print("\ntotal :", el())
